# Final model: how the best submission is built

This notebook explains and reproduces the energy-usage model behind the best leaderboard scores, and writes the next candidate, `candidate_v9.csv`. All the code lives in `final_model.py`; this notebook calls it step by step.

| File | What it is | Leaderboard RMSE |
|---|---|---|
| `best_v2.csv` | MissForest filling + Ridge / gradient boosting | 3.40 |
| `candidate_v4.csv` | per-missing-pattern models + v3 (`energy_model.py`) | 3.37 |
| `candidate_v5_lightgbm.csv` | 0.50 v4 + 0.50 new model | 3.29 |
| `candidate_v6_blend.csv` | 0.30 v4 + 0.40 new model + 0.30 v2 | 3.32 |
| `candidate_v7.csv` | 0.35 v4 + 0.65 new model | 3.27 |
| `candidate_v8.csv` | 0.25 v4 + 0.75 new model | **3.27026** |
| `candidate_v9.csv` | 0.20 v4 + 0.80 new model (averaged over 3 seeds) | not submitted yet |

**The model in three steps**

1. **v4**: one regression per pattern of available inputs, so a row missing `previous_usage` is predicted directly from what it does have, blended with the v3 notebook model.
2. **New model**: LightGBM models first fill in missing inputs, learning from the inputs of both train and test rows (never test answers). A Ridge model plus a small gradient-boosting part then predict energy usage.
3. **Blend**: a weighted average of the two. The weights come from the leaderboard scores (step 3 shows how).

**Environment:** the platform's library versions (`requirements-image.txt`), for example the `datathon-platform-env` environment. No CatBoost is needed.

In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import sklearn, lightgbm
import final_model as F

print('scikit-learn', sklearn.__version__, '| lightgbm', lightgbm.__version__, '| pandas', pd.__version__)
train, test = F.load_data()
print(f'train: {len(train):,} rows | test: {len(test):,} rows')
print('Blank values in test:', test[F.NUM].isna().sum().to_dict())

scikit-learn 1.5.2 | lightgbm 4.5.0 | pandas 2.2.3
train: 8,000 rows | test: 3,000 rows
Blank values in test: {'temperature': 211, 'humidity': 183, 'occupancy': 206, 'previous_usage': 203}


## Step 1: v4, models for each pattern of missing inputs

`energy_model.py` (from your earlier review session) fits **one regression for every combination of available inputs** (16 patterns of the four numeric columns). A test row with `previous_usage` blank is predicted by a model trained without `previous_usage`, instead of from a guessed value. Research on prediction with missing values supports this "pattern submodel" approach.

Each pattern model is a Ridge regression with per-building effects and three daily sine/cosine curves, plus a small tree correction. Training rows are weighted toward the test set's mix of buildings and unusual events. v4 blends these models 75/25 with the v3 notebook model.

The submission uses the exact `candidate_v4.csv` that scored 3.37. The cell below also retrains v4 with these library versions, to show it can be rebuilt from scratch.

In [2]:
v4 = F.v4_predictions(train, test, source='file')
v4_refit = F.v4_predictions(train, test, source='refit')
d = v4_refit - v4
print(f'v4 retrained here vs candidate_v4.csv: mean |difference| {np.abs(d).mean():.3f}, largest {np.abs(d).max():.2f}')

v4 retrained here vs candidate_v4.csv: mean |difference| 0.107, largest 1.70


## Step 2: the new model, LightGBM fills the blanks, then Ridge + trees predict

**2a. Four "imputer" models.** For each of `temperature`, `humidity`, `occupancy` and `previous_usage`, a LightGBM model learns to predict that column from the others (building, type, hour, day, month, the other numeric columns). Three choices make them suit the test set:

- They learn from the **inputs of both train and test rows**. Test answers are never used, because the imputers only predict input columns.
- They train on **extra copies of each row with blank patterns copied from real test rows**, so they learn to fill values even when two or three other columns are missing (common in test, rare in train).
- Rows are **weighted toward the test set's building and event mix**, which has about five times as many unusual rows (heat, rain, crowd surges, usage spikes).

**2b. Energy model.** Blanks are filled with the imputers' predictions. A Ridge regression then uses the v3 notebook features (per-building slopes, event features, occupancy by part of day) plus the imputers' **expected** `previous_usage` and occupancy for each row (with per-building-type versions). It's combined 80/20 with a gradient-boosting model that predicts the change from `previous_usage`.

This model originally used CatBoost for the imputers (`candidate_v5.csv`). LightGBM, which is on the platform list, matched or beat it on held-out rows, and is what scored 3.29.

In [3]:
import json
config = json.loads((F.ROOT / 'model_v5_config.json').read_text())['config']
print('settings:', config)

Model = F._new_model_class()
model = Model(**dict(config, random_seed=527)).fit(train, test)
filled = model.fill(test)
for col in F.NUM:
    blank = test[col].isna()
    print(f'{col:15s} {blank.sum():3d} test blanks filled | typical filled value {filled.loc[blank, col].median():7.1f} '
          f'(observed values median {test[col].median():.1f})')
ridge = model.ridge[-1]
print(f'\nRidge: {ridge.n_features_in_} features, regularization chosen by cross-validation: alpha = {ridge.alpha_:g}')
print(f'Blend inside the new model: {1 - model.tree_blend:.0%} Ridge + {model.tree_blend:.0%} gradient boosting')

settings: {'depth': 5, 'iterations': 1100, 'augment': 3, 'linear_mode': 'notebook', 'distill': True, 'tree_blend': 0.2, 'imputer_weight': 1.5}
temperature     211 test blanks filled | typical filled value    27.8 (observed values median 28.0)
humidity        183 test blanks filled | typical filled value    79.5 (observed values median 78.3)
occupancy       206 test blanks filled | typical filled value    37.0 (observed values median 42.0)
previous_usage  203 test blanks filled | typical filled value    55.2 (observed values median 52.0)

Ridge: 225 features, regularization chosen by cross-validation: alpha = 3.16228
Blend inside the new model: 80% Ridge + 20% gradient boosting


**2c. Averaging three seeds.** The imputers' training copies get random blank patterns, so the model depends a little on the random seed. Averaging three seeds reduces that noise. Seed 527 alone reproduces the model inside your scored files exactly.

In [4]:
seed_preds = {s: F.new_model_predictions(train, test, seeds=(s,)) for s in (527, 528, 529)}
new = np.mean(list(seed_preds.values()), axis=0)

v4_file, v5l = v4, pd.read_csv(F.ROOT / 'candidate_v5_lightgbm.csv').prediction.to_numpy()
inside_v5 = 2 * v5l - v4_file            # candidate_v5_lightgbm = 0.5 v4 + 0.5 new model
print(f'seed 527 vs the model inside candidate_v5_lightgbm.csv: largest difference {np.abs(seed_preds[527] - inside_v5).max():.6f}')
for a, b in [(527, 528), (527, 529), (528, 529)]:
    print(f'seed {a} vs seed {b}: RMS difference {np.sqrt(np.mean((seed_preds[a] - seed_preds[b]) ** 2)):.3f}')

seed 527 vs the model inside candidate_v5_lightgbm.csv: largest difference 0.000000
seed 527 vs seed 528: RMS difference 0.314
seed 527 vs seed 529: RMS difference 0.336
seed 528 vs seed 529: RMS difference 0.309


## Step 3: choosing the blend weights from leaderboard scores

For any weighted average of submitted files whose weights add up to 1, the leaderboard RMSE follows exactly from three things: each file's score, the weights, and how much the files' predictions differ:

$$\text{MSE}\Big(\sum_k w_k\,p_k\Big) = \sum_k w_k\,\text{MSE}_k - \tfrac{1}{2}\sum_{j,k} w_j w_k\,\overline{(p_j-p_k)^2}$$

All six scored files are mixes of the same three ingredients (v2, v4 and the new model). So the cell below finds the ingredient errors that best match all six scores, then searches for the best mix. The scores are rounded, and the leaderboard may only score part of the test set, so treat the result as a guide, not a guarantee.

In [5]:
from scipy.optimize import least_squares, minimize

v2 = pd.read_csv(F.ROOT / 'best_v2.csv').prediction.to_numpy()
ingredients = np.array([v2, v4_file, inside_v5])               # v2, v4, new model (seed 527)
D = np.array([[np.mean((a - b) ** 2) for b in ingredients] for a in ingredients])
scored = {'best_v2': ([1, 0, 0], 3.40), 'candidate_v4': ([0, 1, 0], 3.37), 'candidate_v5_lightgbm': ([0, .5, .5], 3.29),
          'candidate_v6_blend': ([.3, .3, .4], 3.32), 'candidate_v7': ([0, .35, .65], 3.27), 'candidate_v8': ([0, .25, .75], 3.27026)}
mse = lambda w, M: np.asarray(w, float) @ M - 0.5 * np.asarray(w, float) @ D @ np.asarray(w, float)
M = least_squares(lambda M: [np.sqrt(max(mse(w, M), 1e-9)) - s for w, s in scored.values()], x0=np.array([11.5, 11.4, 10.8])).x

table = pd.DataFrame({'weights v2 / v4 / new': [str(w) for w, _ in scored.values()],
                      'leaderboard': [s for _, s in scored.values()],
                      'fitted': [np.sqrt(mse(w, M)) for w, _ in scored.values()]}, index=list(scored))
display(table.round(4))
print('implied score of each ingredient alone: v2 %.3f | v4 %.3f | new model %.3f' % tuple(np.sqrt(M)))
for w in [(0, .25, .75), (0, .2, .8), (0, .15, .85)]:
    print(f'mix v2 {w[0]:.2f} / v4 {w[1]:.2f} / new {w[2]:.2f}: estimated {np.sqrt(mse(w, M)):.4f}')

,weights v2 / v4 / new,leaderboard,fitted
best_v2,"[1, 0, 0]",3.4000,3.4072
candidate_v4,"[0, 1, 0]",3.3700,3.3728
candidate_v5_lightgbm,"[0, 0.5, 0.5]",3.2900,3.2899
candidate_v6_blend,"[0.3, 0.3, 0.4]",3.3200,3.2968
candidate_v7,"[0, 0.35, 0.65]",3.2700,3.2788
candidate_v8,"[0, 0.25, 0.75]",3.2703,3.2750


implied score of each ingredient alone: v2 3.407 | v4 3.373 | new model 3.278
mix v2 0.00 / v4 0.25 / new 0.75: estimated 3.2750
mix v2 0.00 / v4 0.20 / new 0.80: estimated 3.2742
mix v2 0.00 / v4 0.15 / new 0.85: estimated 3.2741


**Reading this:** the best mix sits around 15–20% v4 and 80–85% new model, but the curve is very flat: going from v8 (0.25 / 0.75) to 0.20 / 0.80 is estimated to gain only about 0.001. v2 gets almost no weight; adding it (v6) made the score worse.

## Step 4: write `candidate_v9.csv`

v9 = **0.20 × v4 + 0.80 × new model (average of seeds 527, 528, 529)**. Expect roughly the same as v8 (about 3.27), possibly a little better. The seed averaging slightly improved the test-blanks check on held-out rows at this weight.

In [6]:
v9 = F.blend(v4_file, new, 0.80)
out_path = F.ROOT / 'candidate_v9.csv'
if out_path.exists():
    print('candidate_v9.csv already exists; not overwritten')
    v9_file = pd.read_csv(out_path)
else:
    v9_file = F.write_submission(out_path, test, v9)
    print('wrote candidate_v9.csv')

sample = pd.read_csv('/Users/ethanis/Downloads/sample_submission (1).csv') if __import__('os').path.exists('/Users/ethanis/Downloads/sample_submission (1).csv') else None
v8 = pd.read_csv(F.ROOT / 'candidate_v8.csv').prediction.to_numpy()
print(f'rows {len(v9_file):,} | columns {list(v9_file.columns)} | blanks {v9_file.prediction.isna().sum()}'
      + (f' | same id order as sample_submission: {bool((v9_file.id.values == sample.id.values).all())}' if sample is not None else ''))
print(f'RMS difference from candidate_v8.csv (3.27026): {np.sqrt(np.mean((v9_file.prediction.to_numpy() - v8) ** 2)):.3f}')

wrote candidate_v9.csv
rows 3,000 | columns ['id', 'prediction'] | blanks 0 | same id order as sample_submission: True
RMS difference from candidate_v8.csv (3.27026): 0.149


## Notes

- **What didn't help** (tested on held-out rows or on the leaderboard):
  - adding v2 to the blend;
  - event features on their own (v3 scored 3.45);
  - spline curves, linear trees or a neural network as extra ingredients;
  - expected values for all four inputs.
- **Platform submission:** the template runs a notebook plus a `model.pkl` with no `train.csv`. Everything here uses libraries from `requirements-image.txt`, so the fitted v4 and new-model objects can be saved with `joblib` and loaded by the prediction notebook. The prediction notebook needs `energy_model.py`, `final_model.py` and `experiments/round2_models.py` alongside it.
- **Leaderboard caution:** the scores don't fit the formula perfectly, so the leaderboard may only score part of the test set. Small differences (about 0.005) may not hold in a final ranking on the full test set.